# Scraping — Démo

Film cible : **Spider-Man: Brand New Day**
- Wikipedia (anglais) : https://en.wikipedia.org/wiki/Spider-Man:_Brand_New_Day
- AlloCiné : https://www.allocine.fr/film/fichefilm_gen_cfilm=276608.html

Objectif : passer d'une page HTML brute à des données structurées, en observant à chaque étape ce qu'on récupère réellement.

In [1]:
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "ProjetCineData-Formation/1.0 (usage pedagogique)"}

URL_WIKIPEDIA = "https://en.wikipedia.org/wiki/Spider-Man:_Brand_New_Day"
URL_ALLOCINE = "https://www.allocine.fr/film/fichefilm_gen_cfilm=276608.html"

## 1. Télécharger et regarder le HTML brut

Avant d'écrire le moindre sélecteur, on regarde ce qu'on a vraiment reçu.

In [2]:
response = requests.get(URL_WIKIPEDIA, headers=HEADERS, timeout=10)
response.raise_for_status()

print(f"Code statut : {response.status_code}")
print(f"Taille de la page : {len(response.text)} caractères")
print(response.text[:500])  # les 500 premiers caractères, pour voir à quoi ça ressemble

Code statut : 200
Taille de la page : 1533995 caractères
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientp


## 2. Trouver l'infobox avec BeautifulSoup

Sur Wikipedia, les informations structurées d'un film (réalisateur, durée, casting...) sont dans un tableau `<table class="infobox">`, en haut à droite de l'article. On le repère avec `find`.

**Astuce pour retrouver ce genre de structure sur une autre page :** clic droit sur l'élément dans le navigateur → "Inspecter" → repérer la balise et sa classe CSS.

In [10]:
soup = BeautifulSoup(response.text, "html.parser")
infobox = soup.find("table", class_="infobox")

print(infobox.prettify()[:1000])  # aperçu du HTML de l'infobox

<table about="#mwt9" class="infobox vevent" id="mwEA">
 <tbody>
  <tr>
   <th class="infobox-above summary" colspan="2" style="font-size: 125%; font-style: italic;">
    Spider-Man: Brand New Day
   </th>
  </tr>
  <tr>
   <td class="infobox-image" colspan="2">
    <span class="mw-default-size" data-mw='{"caption":"A close-up of Spider-Man, with a reflection of MJ in the eye of his mask."}' typeof="mw:File/Frameless">
     <a class="mw-file-description" href="https://en.wikipedia.org/wiki/File:Spider-Man_Brand_New_Day_poster.jpg" title="A close-up of Spider-Man, with a reflection of MJ in the eye of his mask.">
      <img alt="A close-up of Spider-Man, with a reflection of MJ in the eye of his mask." class="mw-file-element mw-file-upright" data-file-height="387" data-file-type="bitmap" data-file-width="258" decoding="async" height="375" loading="lazy" resource="https://en.wikipedia.org/wiki/File:Spider-Man_Brand_New_Day_poster.jpg" src="//thumb.wikimedia.org/wikipedia/en/thumb/9/9a/Spi

## 3. Extraire une seule ligne, pour comprendre la structure

Chaque ligne (`<tr>`) contient un label (`<th>`) et une valeur (`<td>`).

In [11]:
premiere_ligne_utile = None
for ligne in infobox.find_all("tr"):
    if ligne.find("th") and ligne.find("td"):
        premiere_ligne_utile = ligne
        break

label = premiere_ligne_utile.find("th").get_text(" ", strip=True)
valeur = premiere_ligne_utile.find("td").get_text(" ", strip=True)
print(f"{label} : {valeur}")

Directed by : Destin Daniel Cretton


## 4. Généraliser à toutes les lignes

Certains champs (ex. les scénaristes) contiennent plusieurs valeurs sous forme de liste à puces (`<li>`) plutôt qu'un simple texte — il faut le prévoir.

In [12]:
def extraire_infobox(html):
    soup = BeautifulSoup(html, "html.parser")
    infobox = soup.find("table", class_="infobox")
    if infobox is None:
        return {}

    donnees = {}
    for ligne in infobox.find_all("tr"):
        label_cell = ligne.find("th")
        valeur_cell = ligne.find("td")
        if label_cell is None or valeur_cell is None:
            continue

        label = label_cell.get_text(" ", strip=True)
        items = valeur_cell.find_all("li")
        if items:
            valeur = "; ".join(item.get_text(" ", strip=True) for item in items)
        else:
            valeur = valeur_cell.get_text(" ", strip=True)

        donnees[label] = valeur
    return donnees


donnees_film = extraire_infobox(response.text)
for label, valeur in donnees_film.items():
    print(f"{label} : {valeur}")

Directed by : Destin Daniel Cretton
Written by : Chris McKenna; Erik Sommers
Based on : Stan Lee; Steve Ditko
Produced by : Kevin Feige; Amy Pascal; Avi Arad; Rachel O'Connor
Starring : Tom Holland; Zendaya; Sadie Sink; Jacob Batalon; Jon Bernthal; Florence Pugh; Tramell Tillman; Marisa Tomei; Mark Ruffalo
Cinematography : Brett Pawlak
Edited by : Nat Sanders; Gina Sansom; Harry Yoon
Music by : Michael Giacchino
Production companies : Columbia Pictures; Marvel Studios; Pascal Pictures
Distributed by : Sony Pictures Releasing
Release dates : July 27, 2026 ( 2026-07-27 ) ( Dolby Theatre ); July 31, 2026 ( 2026-07-31 ) (United States)
Running time : 145 minutes [ 1 ]
Country : United States
Language : English
Budget : $225 million [ 2 ]
Box office : $2.479 billion [ 3 ] [ 4 ]


In [ ]:
contenu = soup.find("div", class_="mw-parser-output")

for p in contenu.find_all("p", recursive=False):
    texte = p.get_text(" ", strip=True)
    if texte:
        print(texte[:400])
        break